# Example: predict the science-arm sky spectrum from minimal inputs

This notebook shows how to use a trained ensemble to predict the sky
spectrum at the science pointing from the smallest possible per-row input:

1. The two sky-arm decomposition coefficient vectors (from the QP
   decomposition, one per sky arm), and
2. Pointing metadata that cannot be derived from anything else:
    - `obstime_mjd` (UT MJD),
    - `sci_ra`, `sci_dec` (science pointing, degrees),
    - `sky_near_ra`, `sky_near_dec` (near sky-arm pointing, degrees),
    - `sky_far_ra`, `sky_far_dec` (far sky-arm pointing, degrees).

Everything else (astrometry, moon/sun ephemerides, van Rhijn slant-path
factors, ecliptic geometry, physics-prior moon-scatter proxies, the
space-weather indices, and the Phase-A' interaction features — 39 context
features per arm in total) is computed inside
`mlp_predictor.inference.predict_sky_from_minimal_inputs`.

**Prerequisite**: run the training notebook
`notebook_sky_interpolation_triplet_dual_encoder_group_mlp_split_zodi_module.ipynb`
to fit the ensemble.  The save cell at the end of that notebook writes the
trained ensemble to a `.pt` archive that this notebook then loads.

## 1. Load the trained ensemble

In [ ]:
import os
os.environ.setdefault("LVMCORE_DIR", "/Users/droryn/prog/lvm/lvmcore")

import numpy as np
from mlp_predictor import config, serialization, inference

cfg = config.PipelineConfig()
ENSEMBLE_PATH = f"{cfg.data.decomp_data_root}/mlp_ensemble_split_zodi_current.pt"
print(f"loading ensemble from: {ENSEMBLE_PATH}")

ensemble = serialization.load_ensemble(ENSEMBLE_PATH)
print(f"loaded {len(ensemble['members'])}-seed ensemble; "
      f"deployed config knobs:")
for k in ('moon_zodi_mode', 'alpha_ctx_groups', 'flux_mse_groups',
         'zodi_ctx_restriction', 'continuum_ctx_restriction'):
    _v = ensemble['config'].get(k)
    print(f"  {k}: {_v}")
print(f"predicted coefficient dim: {len(ensemble['coef_names'])}")
print(f"context features per arm: {len(ensemble['ctx_names'])}")

## 2. Required per-row inputs

The prediction function needs the following minimal set per row.  Angles
are ICRS degrees; MJD is UT.

| input | shape | meaning |
|---|---|---|
| `obstime_mjd`  | `(n_rows,)`   | MJD (UT) of the exposure — sole source of all time features and moon/sun ephemerides |
| `sci_ra`, `sci_dec`         | `(n_rows,)` each | science-fiber pointing (deg) |
| `sky_near_ra`, `sky_near_dec` | `(n_rows,)` each | *near* sky-arm pointing (deg) — closer of the two sky arms to the science pointing |
| `sky_far_ra`, `sky_far_dec`   | `(n_rows,)` each | *far* sky-arm pointing (deg) |
| `coef_near` | `(n_rows, n_coef=433)` | decomposition coefficients from the near sky-arm spectrum |
| `coef_far`  | `(n_rows, n_coef=433)` | decomposition coefficients from the far  sky-arm spectrum |

**Optional overrides** (all default to sensible values if omitted):

| input | default | meaning |
|---|---|---|
| `moon_phase_deg` | computed via astropy | moon phase (0=new, 180=full).  Providing this manually avoids one astropy roundtrip. |
| `sky_near_label`, `sky_far_label` | `'SKYE'` / `'SKYW'` | east/west assignment for the single `ew` ctx feature.  Impact on predictions is small. |

**Not required** (derived internally): everything else in the 39-feature
context — `alt`, `az_{sin,cos}`, `airmass`, `moon_alt`, `moon_sep`,
`moon_phase_{sin,cos}`, `moon_az_{sin,cos}`, `sun_{sep,alt}`,
`sun_az_{sin,cos}`, `sci_sep`, `vanrhijn_{87km,95km,285km}`,
`obstime_{day,lunation,year}_{sin,cos}`, `f107`, `f107_81d`, `kp`,
`ecl_beta_deg`, `ecl_lon_{sin,cos}`, `zodi_log10_v`, `moon_fli`,
`moon_up_smooth`, `moon_airmass_up`, `moon_signal_proxy`,
`moon_fli_x_phase_cos`, `moon_sig_x_lon_{cos,sin}`.

See intro §3.4.5 in the training notebook for how each of those enters the
model.

## 3. Build example inputs and predict

Below we synthesise a small batch of inputs and run the prediction.  In a
real deployment the `coef_near` / `coef_far` arrays come from the sky-arm
decomposition run on the current exposure; here we sample from the corpus
so the numbers stay realistic.

In [ ]:
from astropy.io import fits

# Grab a handful of aligned rows from the training corpus so the sky-arm
# coef vectors are physically realistic.  Any set of rows works.
with fits.open(cfg.data.input_fits_meta) as hdul:
    meta = hdul['META'].data
    n_meta = len(meta)
    idx = np.arange(4)  # first four rows of the corpus
    # OBSTIME is stored as ISO-format strings in the META table; convert
    # to UT MJD via astropy.time.
    from astropy.time import Time
    obstime_mjd  = Time(np.asarray(meta['OBSTIME'][idx]).astype(str),
                       format='isot', scale='utc').mjd
    sci_ra       = np.asarray(meta['SCI_RA'][idx], dtype=np.float64)
    sci_dec      = np.asarray(meta['SCI_DEC'][idx], dtype=np.float64)
    sky_near_ra  = np.asarray(meta['SKY_NEAR_RA'][idx], dtype=np.float64)
    sky_near_dec = np.asarray(meta['SKY_NEAR_DEC'][idx], dtype=np.float64)
    sky_far_ra   = np.asarray(meta['SKY_FAR_RA'][idx], dtype=np.float64)
    sky_far_dec  = np.asarray(meta['SKY_FAR_DEC'][idx], dtype=np.float64)

with fits.open(cfg.data.coef_fits('sky1')) as hdul:
    coef_near = np.asarray(hdul['COEF'].data[idx].tolist(), dtype=np.float32)
with fits.open(cfg.data.coef_fits('sky2')) as hdul:
    coef_far  = np.asarray(hdul['COEF'].data[idx].tolist(), dtype=np.float32)

print(f"batch: n_rows={len(idx)}, coef shape={coef_near.shape}, {coef_far.shape}")
print(f"sci pointing (row 0): ra={sci_ra[0]:.3f} dec={sci_dec[0]:.3f} MJD={obstime_mjd[0]:.4f}")
print(f"near arm (row 0):     ra={sky_near_ra[0]:.3f} dec={sky_near_dec[0]:.3f}")
print(f"far  arm (row 0):     ra={sky_far_ra[0]:.3f} dec={sky_far_dec[0]:.3f}")

In [ ]:
# Run the ensemble prediction (auto-computes all 39 ctx features and
# averages across the 10 seeds).
result = inference.predict_sky_from_minimal_inputs(
    ensemble,
    obstime_mjd=obstime_mjd,
    sci_ra=sci_ra, sci_dec=sci_dec,
    sky_near_ra=sky_near_ra, sky_near_dec=sky_near_dec,
    sky_far_ra=sky_far_ra,   sky_far_dec=sky_far_dec,
    coef_near=coef_near, coef_far=coef_far,
)
print("result keys:", list(result.keys()))
for k in ('coef', 'coef_std', 'confidence'):
    v = result[k]
    print(f"  {k}: shape={v.shape} dtype={v.dtype}")

## 4. Return-value structure

`predict_sky_from_minimal_inputs` returns a dict with the following keys:

| key | shape | meaning |
|---|---|---|
| `coef`         | `(n_rows, 433)` `float32` | **The prediction.** Ensemble-mean science-arm decomposition coefficients, already multiplied by the Jensen per-coefficient bias-lift (§3.6.6 of the training doc).  Reconstruct the physical flux spectrum by matrix-multiplying with the corresponding basis matrix from `SkyDecompLSFSurfaceIterative` (see §5 below). |
| `coef_std`     | `(n_rows, 433)` `float32` | **Per-coefficient epistemic uncertainty.**  Standard deviation across the 10 ensemble seeds — a first-order proxy for the model's confidence in each individual coefficient. |
| `confidence`   | `(n_rows,)` `float32` in (0, 1] | **Row-level confidence score.**  Defined as $1 / (1 + \mathrm{median}_k \ (\sigma_k / |c_k|))$ across the coefficient dimension.  1.0 = perfect seed agreement; ~0.5 = median relative std of ~100%; ~0.1 = the seeds disagree wildly.  See §4.1 below for how to read it. |
| `coef_names`   | list of 433 strings | Coefficient identifiers matching the columns of `coef` and `coef_std`, e.g. `OH_004`, `Moon_bs02`, `Zodi_bs01`. |
| `triplet`      | dict                | The 39-dim per-arm context that was fed to the model, plus the raw RA/Dec/MJD, useful for debugging or feeding a second predictor. |

Pass `return_per_seed=True` to additionally get a `per_seed` key with the
full `(10, n_rows, 433)` per-seed prediction cube.

### 4.1 Reading the confidence score

The scalar `confidence` in each row is a monotone rescaling of the median
*relative* seed spread across the 433 coefficient dimensions:

$$\mathrm{confidence}(r) \;=\; \frac{1}{1 + \mathrm{median}_k \bigl( \sigma_k(r) / |\bar{c}_k(r)| \bigr)}.$$

Practical reading:

* **`confidence \ge 0.9`** — the 10 seeds agree to better than ~10 % relative
  on the median coefficient; the model considers the row well-covered by
  the training distribution.  Typical for moon-down / low-airmass rows.
* **`0.6 \le confidence < 0.9`** — moderate seed spread.  The row sits near
  the edge of the training distribution or in a regime where the physics
  itself is variable (bright-moon-close, close_zodi, gravity-wave-rich
  nights).  The prediction is still usable but the per-pixel error bar is
  wider.
* **`confidence < 0.6`** — the seeds actively disagree.  The row is in a
  regime the model has not seen enough of during training, or the input
  coefficients are outside the coverage of the training corpus.  Treat the
  prediction as a rough guess and consider flagging it downstream.

For downstream code that needs a full covariance rather than a scalar
summary, use `coef_std` directly — it gives the per-coefficient 1-sigma
spread across the ensemble, on the same physical scale as `coef`.

In [ ]:
# Show the confidence per row and highlight which coefficients are
# most/least uncertain.
for r in range(result['coef'].shape[0]):
    c   = result['coef'][r]
    std = result['coef_std'][r]
    conf = float(result['confidence'][r])
    # Relative std, guarding against tiny |c|.
    rel = std / np.maximum(np.abs(c), 1e-12)
    order = np.argsort(rel)
    worst = order[-3:][::-1]
    best  = order[:3]
    print(f"row {r}: confidence = {conf:.3f}")
    print(f"   most uncertain coefs: ", end='')
    print(', '.join(f"{result['coef_names'][k]}={c[k]:+.3g}±{std[k]:.2g}" for k in worst))
    print(f"   most certain coefs:   ", end='')
    print(', '.join(f"{result['coef_names'][k]}={c[k]:+.3g}±{std[k]:.2g}" for k in best))

## 5. Optional: reconstruct the flux spectrum

The prediction is a coefficient vector; the flux spectrum is
$f(\lambda) = A \hat{\mathbf{c}}$ where $A$ is the design matrix from
the decomposition side (§1.3 of the training doc).  We reconstruct that
here on the deployed wavelength grid using the same class the training
pipeline uses to build the flux-MSE basis (§3.6.1).

This step is optional: many consumers of the ML pipeline only need the
coefficient vector to subtract from a science exposure at the fit level.
Skip this section if you already have the design matrix cached.

In [ ]:
from astropy.io import fits
from sky_decomp.lsf_surface_iterative import SkyDecompLSFSurfaceIterative
from mlp_predictor import wavelengths as _wave_mod
from mlp_predictor.data import _infer_base_dir_for_reconstruction

# Wavelength grid from the notebook's fiducial input FITS.
with fits.open(cfg.data.input_fits_for_basis) as hdul:
    wave = np.asarray(hdul['WAVE'].data, dtype=np.float64)

# Spline-knot counts inferred from the deployed coefficient names.
N_MOON_KNOTS, SPLIT_ZODI, N_ZODI_KNOTS = _wave_mod.infer_spline_knots(ensemble['coef_names'])
print(f"n_moon_knots={N_MOON_KNOTS}, split_zodi={SPLIT_ZODI}, n_zodi_knots={N_ZODI_KNOTS}")

# Build a reconstruction model on the fiducial LSF (sigma=1.0).  A per-row
# LSF surface state can be passed instead if the deployed sky-arm decomps
# carry LSF_COEF / LSF_KNOTS / LSF_META HDUs; sigma=1.0 is a reasonable
# broadband proxy for continuum reconstruction.
recon = SkyDecompLSFSurfaceIterative(
    wave, lsf_sigma=1.0, n_spline_knots=N_MOON_KNOTS,
    base_dir=_infer_base_dir_for_reconstruction(),
    palace_oh_suffix='_joint_v2_updated',
    palace_diffuse_suffix='_joint_native_adam_invsky_p2_10000iter',
    split_zodi=SPLIT_ZODI, n_zodi_spline_knots=N_ZODI_KNOTS,
)
mats = recon._assemble_refined_matrices()

def _sum_components(comps):
    # Sum per-family fluxes; 'zodi' is only present when split_zodi=True.
    total = np.zeros_like(np.asarray(comps['oh'], dtype=np.float64))
    for key in ('oh', 'moon', 'zodi', 'diffuse', 'atom', 'orc', 'o2'):
        arr = comps.get(key)
        if arr is not None:
            total = total + np.asarray(arr, dtype=np.float64)
    return total

# Row 0 reconstruction (mean coefficients + ensemble std envelope).
coef0    = result['coef'][0].astype(np.float64)
coef0_lo = coef0 - result['coef_std'][0].astype(np.float64)
coef0_hi = coef0 + result['coef_std'][0].astype(np.float64)
flux_mean = _sum_components(recon._components_from_coef(coef0, mats))
flux_lo   = _sum_components(recon._components_from_coef(np.maximum(coef0_lo, 0), mats))
flux_hi   = _sum_components(recon._components_from_coef(np.maximum(coef0_hi, 0), mats))

print(f"reconstructed spectrum: n_lambda={flux_mean.size}, mean flux range=[{flux_mean.min():.3g}, {flux_mean.max():.3g}]")

In [ ]:
# Plot the reconstructed sky spectrum with a shaded ensemble-uncertainty band.
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=wave, y=flux_hi, mode='lines',
                          line=dict(color='rgba(30, 30, 200, 0.0)'),
                          showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=wave, y=flux_lo, mode='lines',
                          line=dict(color='rgba(30, 30, 200, 0.0)'),
                          fill='tonexty', fillcolor='rgba(30, 30, 200, 0.15)',
                          name='±1σ ensemble spread'))
fig.add_trace(go.Scatter(x=wave, y=flux_mean, mode='lines',
                          line=dict(color='rgb(30, 30, 200)', width=1.0),
                          name='predicted sky flux (row 0)'))
fig.update_layout(xaxis_title='wavelength [Å]',
                   yaxis_title='flux (native decomp units)',
                   height=400, margin=dict(l=60, r=20, t=40, b=50),
                   title=f"row 0: confidence={result['confidence'][0]:.3f}")
fig.show()

## 6. Where to look next

- **`mlp_predictor.inference.build_triplet_from_pointings`** — the ctx
  builder called under the hood.  Returns the 39-dim per-arm context if
  you want to feed it to `predict_sci_coefficients_default` directly.
- **`mlp_predictor.trainer.predict_sci_coefficients_default(artifact, ...)`** —
  the low-level predictor.  Bypasses the ensemble averaging so you can
  probe per-seed predictions if you want a specific seed's output.
- **Training notebook §3.4.5** — a table of which of the 39 context
  features enter which specialised branch of the model.
- **Training notebook §3.4.3** — the derivation of the Phase-F additive
  moon-zodi coupling that the ensemble uses; explains why the additive
  residual is zero at $t=0$ and where the moon/zodi predictions get their
  cross-group information.
- **`sky_decomp.lsf_surface_iterative.SkyDecompLSFSurfaceIterative`** —
  the same class used above to reconstruct the flux spectrum.  Use its
  `_components_from_coef` on your predicted `coef` to get the per-family
  breakdown (moon, zodi, OH lines, atomic, diffuse continuum, O₂) if you
  need to subtract only a subset.